# Project : AI-Powered Code Suggester

By Akshat Kesharwani | Free tools only (Google Colab + free-tier APIs)

## Problem Statement

GitHub Copilot showed that a model trained on real code can genuinely speed up coding by suggesting completions as you type. CodeLlama-7B needs paid GPUs to fine-tune, so here we use DistilGPT2 instead -- much smaller, still learns the same kind of code-completion skill (predict what comes next given the code so far), and trains fine on the free Colab GPU. (Note: we originally used CodeT5-small here, but its tokenizer hit a bug on the version of `transformers` Colab currently ships with -- DistilGPT2 uses the plain GPT-2 tokenizer, which doesn't have that issue.)

### SECTION 1: Install & Import Libraries

In [1]:
!pip install -q transformers datasets accelerate evaluate openai

import os
import torch
import pandas as pd
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    TrainingArguments, Trainer, DataCollatorForLanguageModeling
)

# ---- Small AI helper: call Groq (free) ----
# Only using Groq here -- it's free, fast, and just needs an API key,
# no extra setup like Ollama needs.
# Get a free key at https://console.groq.com/keys
# (Update: as of June 2026 Groq retired the old Llama-3.3-70B model,
# the current one to use is "openai/gpt-oss-120b".)
from getpass import getpass
from openai import OpenAI

groq_key = getpass("Enter your Groq API key: ")
groq_client = OpenAI(base_url="https://api.groq.com/openai/v1", api_key=groq_key)

def call_llm(prompt, max_tokens=300):
    response = groq_client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_tokens
    )
    return response.choices[0].message.content

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.3 MB/s eta 0:00:00
Enter your Groq API key: ··········


### SECTION 2: Build Python Functions Corpus (robust, mostly offline)

In [2]:
# quick heads up: "codeparrot/codecomplex" is actually a pretty small
# dataset (a few thousand rows, made for complexity classification, not
# 50K+ completion pairs) so trying to sample 50,000 rows from it throws an
# error, and the column names don't always match either. easiest fix: just
# grab real Python functions straight from the standard library that's
# already installed on this machine (no internet needed, always works),
# then try to add more from a bigger dataset online if that's reachable.
import inspect
import importlib
import warnings
warnings.filterwarnings("ignore")

def collect_stdlib_functions(min_len=80, max_len=3000):
    functions = []
    modules_to_scan = ["os", "re", "json", "itertools", "functools", "collections",
                        "string", "math", "random", "statistics", "datetime", "shutil",
                        "csv", "textwrap", "pathlib", "urllib.parse", "argparse",
                        "logging", "unittest", "heapq", "bisect", "queue", "difflib"]
    for mod_name in modules_to_scan:
        try:
            mod = importlib.import_module(mod_name)
        except Exception:
            continue
        for _, obj in inspect.getmembers(mod, inspect.isfunction):
            try:
                src = inspect.getsource(obj)
            except (TypeError, OSError):
                continue
            if min_len <= len(src) <= max_len:
                functions.append(src)
        for _, cls_obj in inspect.getmembers(mod, inspect.isclass):
            for _, method in inspect.getmembers(cls_obj, inspect.isfunction):
                try:
                    src = inspect.getsource(method)
                except (TypeError, OSError):
                    continue
                if min_len <= len(src) <= max_len:
                    functions.append(src)
    return list(dict.fromkeys(functions))  # dedup, preserve order

code_samples = collect_stdlib_functions()
print(f"[dataset] Collected {len(code_samples)} functions from the local Python standard library.")

# Best-effort top-up with a larger remote dataset (skipped cleanly if unavailable).
try:
    print("[dataset] Attempting to top up with bigcode/the-stack-smol (Python subset)...")
    extra = load_dataset("bigcode/the-stack-smol", data_dir="data/python", split="train")
    extra_code = [c for c in extra["content"] if 50 < len(c) < 3000]
    code_samples.extend(extra_code[:20000])
    print(f"[dataset] Added {len(extra_code[:20000])} extra functions from the remote dataset.")
except Exception as e:
    print(f"[dataset] Remote top-up skipped ({type(e).__name__}: {e}) -- continuing with stdlib-only corpus.")

df = pd.DataFrame({"code": code_samples})
df = df[df["code"].str.len() > 50].sample(n=min(50000, len(df)), random_state=42, replace=len(df) < 50000).reset_index(drop=True)

def split_prefix_target(code):
    cut = int(len(code) * 0.7)
    return code[:cut], code[cut:]

df["prefix"], df["target"] = zip(*df["code"].map(split_prefix_target))
print("Training functions:", len(df))

[dataset] Collected 998 functions from the local Python standard library.
[dataset] Attempting to top up with bigcode/the-stack-smol (Python subset)...


README.md:   0%|          | 0.00/3.30k [00:00<?, ?B/s]

[dataset] Remote top-up skipped (DatasetNotFoundError: Dataset 'bigcode/the-stack-smol' is a gated dataset on the Hub. You must be authenticated to access it.) -- continuing with stdlib-only corpus.
Training functions: 998


### SECTION 3: Tokenize

In [3]:
# switched from CodeT5-small to distilgpt2 here -- CodeT5's tokenizer files
# hit a known bug on the transformers version Colab currently has
# preinstalled (throws "Input must be a List[Union[str, AddedToken]]").
# distilgpt2 uses the plain GPT-2 tokenizer, which doesn't have that problem
# and is the same tokenizer already working fine in the other projects.
model_checkpoint = "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
tokenizer.pad_token = tokenizer.eos_token  # gpt2 has no pad token by default, reuse eos

dataset = Dataset.from_pandas(df[["code"]])
dataset = dataset.train_test_split(test_size=0.05, seed=42)

def tokenize_fn(batch):
    texts = [c + tokenizer.eos_token for c in batch["code"]]
    return tokenizer(texts, truncation=True, max_length=256, padding="max_length")

tokenized = dataset.map(tokenize_fn, batched=True, remove_columns=dataset["train"].column_names)

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Map:   0%|          | 0/948 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

### SECTION 4: Tokenizer Optimization for Lower Latency

In [4]:
# Truncate to the smallest max_length that still covers 95th percentile
# code length -- cuts padding waste and inference latency.
code_lengths = [len(tokenizer.encode(c)) for c in df["code"].sample(min(2000, len(df)), random_state=1)]
import numpy as np
p95 = int(np.percentile(code_lengths, 95))
print(f"95th percentile code token length: {p95} (used to right-size max_length)")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1066 > 1024). Running this sequence through the model will result in indexing errors


95th percentile code token length: 834 (used to right-size max_length)


### SECTION 5: Load & Fine-Tune DistilGPT2 for Code Completion

In [5]:
model = AutoModelForCausalLM.from_pretrained(model_checkpoint)
data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

use_fp16 = torch.cuda.is_available()  # fp16 only works with a real GPU -- Runtime > Change runtime type > T4 GPU
print("GPU available:", use_fp16)

training_args = TrainingArguments(
    output_dir="./code-suggester-checkpoints",
    eval_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=4,      # kept small so free Colab doesn't run out of RAM
    per_device_eval_batch_size=4,
    num_train_epochs=3,
    weight_decay=0.01,
    gradient_accumulation_steps=4,   # makes up for the smaller batch size above
    gradient_checkpointing=True,     # uses less memory, trains a little slower
    save_strategy="epoch",
    save_total_limit=1,
    fp16=use_fp16,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    data_collator=data_collator,
    processing_class=tokenizer  # newer transformers renamed "tokenizer" to "processing_class"
)

trainer.train()

model.safetensors: reconstructing file:   0%|          |  0.00B /  353MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

GPU available: True


...

Epoch,Training Loss,Validation Loss
1,No log,1.849856
2,No log,1.724654
3,No log,1.691420


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=180, training_loss=1.9516649034288194, metrics={'train_runtime': 99.7174, 'train_samples_per_second': 28.521, 'train_steps_per_second': 1.805, 'total_flos': 185781989670912.0, 'train_loss': 1.9516649034288194, 'epoch': 3.0})

### SECTION 6: Measure Latency Improvement

In [6]:
import time

def measure_latency(prefix, n_runs=20):
    inputs = tokenizer(prefix, return_tensors="pt", truncation=True, max_length=256)
    start = time.time()
    for _ in range(n_runs):
        with torch.no_grad():
            model.generate(
                **{k: v.to(model.device) for k, v in inputs.items()},
                max_new_tokens=64, pad_token_id=tokenizer.eos_token_id
            )
    return (time.time() - start) / n_runs

def split_prefix_target(code):
    cut = int(len(code) * 0.7)
    return code[:cut], code[cut:]

sample_prefix, _ = split_prefix_target(df["code"].iloc[0])
avg_latency = measure_latency(sample_prefix)
print(f"Average generation latency: {avg_latency*1000:.1f} ms per suggestion")

Average generation latency: 584.4 ms per suggestion


### SECTION 7: Save Model

In [7]:
model.save_pretrained("./code-suggester-model")
tokenizer.save_pretrained("./code-suggester-model")
print("Model saved to ./code-suggester-model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to ./code-suggester-model
